In [ ]:
import os, uuid
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

load_dotenv()

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    
llm = init_chat_model(
    "azure_openai:gpt-4o-mini"
)

### How to add semantic search to agent's memory

In [ ]:
from langchain.embeddings import init_embeddings
from langgraph.store.memory import InMemoryStore

# Create store with semantic search enabled
embeddings = init_embeddings(
    "azure_openai:text-embedding-ada-002", 
    openai_api_key=os.getenv("AZURE_TEXT_EMBEDDING_ADA_002_KEY")
)

store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1536,
    }
)

store.put(("user_123", "memories"), "1", {"text": "I love pizza"})
store.put(("user_123", "memories"), "2", {"text": "I am a plumber"})

memories = store.search(
    ("user_123", "memories"), query="I have leak in my pipe", limit=1
)

for memory in memories:
    print(f"Memory: {memory.value['text']}, Score: {memory.score}")
    
# Note: Require Numpy for vector 
# https://langchain-ai.github.io/langgraph/how-tos/memory/semantic-search/